In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.style as style
from statsmodels.tsa.api import seasonal_decompose
import datetime
import calendar
import time
#import pandas.util.testing as tm
import matplotlib.pyplot as mp
import json
import ast
from tqdm import tqdm
style.use('ggplot')

# **Funções**

In [ ]:
def fix_size(dataframe, data_day):
    # Cria dfs fake nas datas inicial e final para homogenizar o tamanho das séries
    data_inicio = str(data_day.replace(day=1))
    res = calendar.monthrange(data_day.year, data_day.month)
    data_fim = str(data_day.replace(day=int(res[1])))

    df = dataframe

    n_unicos = len(df.CodProduto.unique())

    df_ini = df.tail(n_unicos)
    df_ini = df_ini.reset_index()
    df_ini = df_ini.iloc[:, 1:]
    df_ini.DtEmissao = data_inicio
    df_ini.DtEmissao= pd.to_datetime(df_ini.DtEmissao)
    df_ini.CodProduto = df.CodProduto.unique()
    df_ini.QuantdadeItem = 0.0
    df_ini.TpMovi = 'FAKE'
    df_ini.serie = 999
    df_ini.documento = 0
    df_ini.IdFilial = 999
    df_ini.PrecoItem = 0.0
    df_ini.TotalItem = 0.0

    df_end = df.tail(n_unicos)
    df_end = df_end.reset_index()
    df_end = df_end.iloc[:, 1:]
    df_end.DtEmissao = data_fim
    df_end.DtEmissao= pd.to_datetime(df_end.DtEmissao)
    df_end.CodProduto = df.CodProduto.unique()
    df_end.QuantdadeItem = 0.0
    df_end.TpMovi = 'FAKE'
    df_end.serie = 999
    df_end.documento = 0
    df_end.IdFilial = 999
    df_end.PrecoItem = 0.0
    df_end.TotalItem = 0.0

    df = df._append(df_ini, ignore_index=True)
    df = df._append(df_end, ignore_index=True)

    df = df.sort_values(by='DtEmissao')
    df = df.reset_index()
    df = df.iloc[:, 1:]

    return df

from tqdm import tqdm

def process(dataframe, produto):
    df = dataframe
    prod = produto
    dfs = df[df['CodProduto']==prod]
    return dfs

def ind_prod(list, produto):
    lst = list
    prod = produto
    indice = (lst).index(produto)
    return indice

def ind_prod(list, produto):
    lst = list
    prod = produto
    indice = (lst).index(produto)
    return indice

def resample_df(dataframe, temporal):
    # Resample somente para Mensal: temporal='M'
    df = dataframe
    temp = temporal
    df_aux = df.resample(temp, on='DtEmissao', closed='right', label='right').sum(True)
    return df_aux

def reset_index(dataframe):
    df = dataframe
    df = df.reset_index()
    return df

def cria_df_historico(list_dataframes, list_produtos):
    l_df = list_dataframes
    l_prod = list_produtos

    df_aux = l_df[0].iloc[:, 0:1]

    l_str = list(map(str, l_prod))

    blot = [l_df[indice].QuantdadeItem for indice in tqdm(list(range(len(list_produtos))))]
    df_aux = pd.concat([df_aux]+blot, axis=1)
    df_aux.columns = ['DtEmissao']+l_str

    return df_aux

# Removendo duplicatas
def eliminate_doubles(dataframe_compras, lista_fornecedor):
    df = dataframe_compras
    list = lista_fornecedor

    df_aux = df.loc[df.CodProduto.isin(list)]
    df_aux = df_aux.drop_duplicates()
    df_aux = df_aux.reset_index()
    df_aux = df_aux.iloc[:, 1:]

    return df_aux

# Intersecção e diferença
def inter_dif(lst1, lst2):
    intersection = set(lst1).intersection(lst2)
    difference1 = set(lst1).difference(set(lst2))
    difference2 = set(lst2).difference(set(lst1))
    print('total primeira: ', len(lst1))
    print('total segunda: ', len(lst2))
    print('intersecção: ', len(intersection))
    print('primeira - segunda: ', len(difference1))
    print('segunda - primeira', len(difference2))
    return list(intersection), list(difference1), list(difference2)

def imprime_inter_dif(lst1, lst2):
    pp = inter_dif(lst1, lst2)
    interseccao = pp[0]
    nosso_nao_adalberto = pp[1]
    adalberto_nao_nosso = pp[2]
    print('total primeira: ', len(lst1))
    print('total segunda: ', len(lst2))
    print('intersecção: ', len(interseccao))
    print('primeira - segunda: ', len(nosso_nao_adalberto))
    print('segunda - primeira', len(adalberto_nao_nosso))

def flatten_json(nested_json: dict, exclude: list=['']) -> dict:
    """
    Flatten a list of nested dicts.
    """
    out = dict()
    def flatten(x: (list, dict, str), name: str='', exclude=exclude):
        if type(x) is dict:
            for a in x:
                if a not in exclude:
                    flatten(x[a], f'{name}{a}_')
        elif type(x) is list:
            i = 0
            for a in x:
                flatten(a, f'{name}{i}_')
                i += 1
        else:
            out[name[:-1]] = x

    flatten(nested_json)
    return out

# **Main**

In [ ]:
# INPUTS
#nome_atualizacao = 'nov23_mid'
data = datetime.date(2023, 11, 22)
eh_completo = False

#SET UP
dia = data.day
mes = data.month
ano = data.year
s_dia = '{:02d}'.format(data.day)
s_mes = '{:02d}'.format(data.month)
s_ano = str(ano)
s_data = s_ano+ s_mes + s_dia

inicio = datetime.date(ano, mes, 1)

# VENDAS
#!unzip {'vendas_' + s_ano + s_mes + '01_' + s_data + '.zip' }
#import glob
#import os
#files = os.path.join("20*.csv")
#files = glob.glob(files)
#dfcru = pd.concat(map(pd.read_csv, files), ignore_index=True)
#!rm 20*.csv

#EMENDA
df_cru = pd.read_csv('df_vendas' + s_ano + s_mes + '01_'+ s_data +'.csv')
df_cru = df_cru[['id', 'filial', 'timestamp_venda', 'codproduto','id', 'tipo_movimento', 'amount', 'unit_price', 'valor', 'id', 'id']]
df_cru.columns = ['id','filial', 'emissao', 'cod prodt', 'cliente', 'tipo movimento', 'quantidade', 'preco item', 'valor total', 'documento', 'serie']


dfcru_atualizado = df_cru
dfcru_atualizado = dfcru_atualizado.iloc[:, 1:]


# Removendo duplicatas
df = dfcru_atualizado
df = df.drop_duplicates()
df = df.reset_index()
df = df.iloc[:, 1:]
dfcru_atualizado = df

#dfcru_atualizado.columns = ['index','IdFilial','DtEmissao','documento', 'serie', 'CodProduto', 'buyer_id','loja', 'conta', 'doct. orig', 'serie orig.',
#                           'centro custo','tes','cfop' , 'tipo nota','TpMovi', 'QuantdadeItem', 'armazem',  'PrecoItem', 'desconto', 'TotalItem', 'modelo veic', 'tipo veic']

#Renomeando colunas para não quebrar o código
#display(dfcru_atualizado)
lista_origi = ['filial', 'emissao', 'cod prodt', 'cliente', 'tipo movimento', 'quantidade', 'preco item', 'valor total']
lista_troc = ['IdFilial', 'DtEmissao', 'CodProduto', 'buyer_id', 'TpMovi', 'QuantdadeItem', 'PrecoItem', 'TotalItem']
listaX = list(dfcru_atualizado.columns)
df_listaX = pd.DataFrame({'listaX':listaX})
df_listaX = df_listaX.replace(lista_origi, lista_troc)
dfcru_atualizado.columns = list(df_listaX.listaX)
display(dfcru_atualizado)

# Erro na coluna TotalItem - Refazê-la como QuantdadeItem * PrecoItem
#dfcru_atualizado['TotalNota']=dfcru_atualizado['TotalItem']
#dfcru_atualizado['TotalItem']=dfcru_atualizado['QuantdadeItem']*dfcru_atualizado['PrecoItem']

dfcru_atualizado = dfcru_atualizado[['IdFilial','DtEmissao','documento', 'serie', 'CodProduto', 'TpMovi', 'QuantdadeItem','PrecoItem','TotalItem']]
#Removendo espaços no início e fim de produtos:
dfcru_atualizado.CodProduto = dfcru_atualizado.CodProduto.str.strip()

dftodasemnan = dfcru_atualizado.dropna()
dffiltrado = dftodasemnan[(dfcru_atualizado.CodProduto.str.contains('SERV') == False) &
                      (dfcru_atualizado.TpMovi.str.contains('SERVICO') == False)]
# Formatando DtEmissao para datetime

# Função de filtragem
dfcru_atualizado['DtEmissao'] = pd.to_datetime(dfcru_atualizado['DtEmissao'].astype(str), format='%Y-%m-%d')
def filtra_df(dataframe):
  # Corrige data para o formato adequado
  dataframe['DtEmissao'] = pd.to_datetime(dataframe['DtEmissao'].astype(str), format='%Y-%m-%d')

  dfsemnan = dataframe.dropna()

  dffiltrado_vendas = dfsemnan[(dataframe.CodProduto.str.contains('SERV') == False) &
                               (dataframe.TpMovi.str.contains('SERVICO') == False) &
                                (dataframe.TpMovi.str.contains('VENDA') == True)]

  dffiltrado_devolve = dfsemnan[(dataframe.CodProduto.str.contains('SERV') == False) &
                               (dataframe.TpMovi.str.contains('SERVICO') == False) &
                                (dataframe.TpMovi.str.contains('DEVOLUCAO PRODUTO') == True)]

  list_filiN = [101, 104, 108, 117, '0101', '0104', '0108', '0117']
  list_a = list(dffiltrado.IdFilial.unique())
  list_filiB =  list(set(list_a).difference(set(list_filiN)))
  dffiltrado_vendas = dffiltrado_vendas[dffiltrado_vendas.IdFilial.isin(list_filiB)]
  dffiltrado_devolve = dffiltrado_devolve[dffiltrado_devolve.IdFilial.isin(list_filiB)]
  return dffiltrado_vendas, dffiltrado_devolve

# Filtragem em si
dfraw_filtrado = filtra_df(dfcru_atualizado)

dfraw_filtrado_vendas = dfraw_filtrado[0].sort_values('DtEmissao')
dfraw_filtrado_vendas

dfraw_filtrado_devolve = dfraw_filtrado[1]

## Cria histórico do Vendas
# df Cópia do raw
df_copy = dfraw_filtrado_vendas
display(df_copy)
# df extensão do cópia :
# com quantidades 0 na data inicial e final para cada produto
df_ext = fix_size(df_copy, data)

df = df_ext
l_produtos = df.CodProduto.unique()
# dfs_produtos - dataframe de dataframes, cru e separado por produtos
# Já inclui linhas fake com quantidade 0 inicial e final
dfs_produtos = [process(df, produto) for produto in tqdm(l_produtos)]

temporal = 'M'
dfs = dfs_produtos
# dfs_produtosM : dfs_produtos reescalado para M
dfs_produtosM = [resample_df(df, temporal) for df in tqdm(dfs)]

dfs = dfs_produtosM
# dfsM - com índices reiniciados
dfsM = [reset_index(df) for df in tqdm(dfs)]

list_total = list(df_ext.CodProduto.unique())

df_histM = cria_df_historico(dfsM, list_total)
print('vendas')
display(df_histM)

## Cria histórico do Devolve Vendas
# df Cópia do raw
df_copy = dfraw_filtrado_devolve
display(df_copy)
# df extensão do cópia :
# com quantidades 0 na data inicial e final para cada produto
df_ext = fix_size(df_copy, data)

df = df_ext
l_produtos = df.CodProduto.unique()
# dfs_produtos - dataframe de dataframes, cru e separado por produtos
# Já inclui linhas fake com quantidade 0 inicial e final
dfs_produtos = [process(df, produto) for produto in tqdm(l_produtos)]

temporal = 'M'
dfs = dfs_produtos
# dfs_produtosM : dfs_produtos reescalado para M
dfs_produtosM = [resample_df(df, temporal) for df in tqdm(dfs)]

dfs = dfs_produtosM
# dfsM - com índices reiniciados
dfsM = [reset_index(df) for df in tqdm(dfs)]

list_total = list(df_ext.CodProduto.unique())

df_histM_devolve = cria_df_historico(dfsM, list_total)
print('devolve vendas')
display(df_histM_devolve)

# COMPRAS
#Pré processamento de devoluções
#!unzip {'compras_' + s_ano + s_mes + '01_' + s_data + '.zip' }
#import glob
#import os
#files = os.path.join("20*.csv")
#files = glob.glob(files)
#dfcru_c = pd.concat(map(pd.read_csv, files), ignore_index=True)
#!rm 20*.csv

#EMENDA
dfcru_c = pd.read_csv('df_compras' +s_ano + s_mes + '01_'+ s_data +'.csv')
dfcru_c = dfcru_c[['FILIAL', 'EMISSAO_NOTA', 'PRODUTO', 'FORNECEDOR', 'DESCRICAO_TES', 'QUANTIDADE', 'VALOR_UNITARIO', 'VALOR_BRUTO']]
#dfcru_c.columns = ['id','filial', 'emissao', 'cod prodt', 'cliente', 'tipo movimento', 'quantidade', 'preco item', 'valor total', 'documento', 'serie']


#dfcru_compras = dfcru_c.iloc[:, 1:]
dfcru_compras = dfcru_c

# Removendo duplicatas
df = dfcru_compras
df = df.drop_duplicates()
df = df.reset_index()
df = df.iloc[:, 1:]
dfcru_compras = df

#print(dfcru_compras.columns)
#dfcru_compras.columns = ['IdFilial','DtEmissao','documento', 'serie','doc_despesa', 'ser_despesa', 'CodProduto', 'fornecedor','loja', 'conta', 'centro_custo', 'tes',
#                            'cfop','TpMovi', 'QuantdadeItem',   'PrecoItem', 'desconto', 'TotalItem', 'modelo veic', 'tipo veic', 'armazem']


#Renomeando colunas para não quebrar o código
#display(dfcru_atualizado)
lista_origi = ['FILIAL', 'EMISSAO_NOTA', 'PRODUTO', 'FORNECEDOR', 'DESCRICAO_TES', 'QUANTIDADE', 'VALOR_UNITARIO', 'VALOR_BRUTO']
lista_troc = ['IdFilial', 'DtEmissao', 'CodProduto', 'buyer_id', 'TpMovi', 'QuantdadeItem', 'PrecoItem', 'TotalItem']
listaX = list(dfcru_compras.columns)
df_listaX = pd.DataFrame({'listaX':listaX})
df_listaX = df_listaX.replace(lista_origi, lista_troc)
dfcru_compras.columns = list(df_listaX.listaX)
display(dfcru_compras)

#Removendo espaços no início e fim de produtos:
dfcru_compras.CodProduto = dfcru_compras.CodProduto.str.strip()

# Erro na coluna TotalItem - Refazê-la como QuantdadeItem * PrecoItem
dfcru_compras['TotalNota']=dfcru_compras['TotalItem']
dfcru_compras['TotalItem']=dfcru_compras['QuantdadeItem']*dfcru_compras['PrecoItem']

# Pegando só as colunas relevantes
dfcru_compras = dfcru_compras[['IdFilial','DtEmissao','CodProduto', 'TpMovi', 'QuantdadeItem','PrecoItem','TotalItem']]

# Removendo nans
dftodasemnan_compras = dfcru_compras.dropna()


# Formatando DtEmissao para datetime

# Função de filtragem
dfcru_compras['DtEmissao'] = pd.to_datetime(dfcru_compras['DtEmissao'].astype(str), format='%Y-%m-%d')
def filtra_df(dataframe):
  # Corrige data para o formato adequado
  dataframe['DtEmissao'] = pd.to_datetime(dataframe['DtEmissao'].astype(str), format='%Y-%m-%d')

  dfsemnan = dataframe.dropna()

  dffiltrado = dfsemnan[(dataframe.TpMovi.str.contains('DEVOL') == True)|(dataframe.TpMovi.str.contains('RETORNO') == True)]
  list_filiN = [101, 104, 108, 117, '0101', '0104', '0108', '0117']
  list_a = list(dffiltrado.IdFilial.unique())
  list_filiB =  list(set(list_a).difference(set(list_filiN)))
  dffiltrado = dffiltrado[dffiltrado.IdFilial.isin(list_filiB)]

  return dffiltrado

#Filtragem em si
dfraw_filtrado_compras = filtra_df(dfcru_compras)

dfraw_filtrado_compras11 = dfraw_filtrado_compras.sort_values('DtEmissao')
dfraw_filtrado_compras11
display(dfraw_filtrado_compras11)
lista_devolucoes = list(dfraw_filtrado_compras11.CodProduto.unique())
df_dev_doubles = eliminate_doubles(dfraw_filtrado_compras11, list_total)
df_dev_ext = fix_size(df_dev_doubles, data)
list_dev_total = list(df_dev_ext.CodProduto.unique())
df = df_dev_ext
print('compras')
display(df)
print('len dev_total', len(list_dev_total))
print('len total', len(list_total))
print('len devolucoes', len(lista_devolucoes))
dfs_produtos_dev = [process(df, produto) for produto in tqdm(list_dev_total)]
temporal = 'M'
dfs_produtos_devM = [resample_df(df, temporal) for df in tqdm(dfs_produtos_dev)]
dfs_devM = [reset_index(df) for df in tqdm(dfs_produtos_devM)]

df_hist_devM = cria_df_historico(dfs_devM, list_dev_total)
int_dif =  inter_dif(list_total, list_dev_total)
# Lista de produtos para completar com zeros
list_completar = int_dif[1]
# Completando com zeros (fazer uma função depois)

comp = len(df_hist_devM)
lst_zeros = [[0]*comp for produto in list_completar]
df_completa_dev = pd.DataFrame(lst_zeros)
df_completa_dev = df_completa_dev.transpose()
df_completa_dev.columns = list_completar
df_hist_devM_complete = pd.concat([df_hist_devM, df_completa_dev], axis=1)
df_histM.index = df_histM.DtEmissao
df_histM = df_histM.iloc[:, 1:]
df_hist_devM_complete.index = df_hist_devM_complete.DtEmissao
df_hist_devM_complete = df_hist_devM_complete.iloc[:, 1:]
df_vendas_finalM = df_histM - df_hist_devM_complete - df_histM_devolve

df_vendas_finalM = df_histM
display(df_vendas_finalM)

if eh_completo == False:

    # ESTOQUE
    #!unzip {'estoque_' + s_data + '.zip' }
    #import glob
    #import os
    #files = os.path.join("produto*.csv")
    #files = glob.glob(files)
    #intermed = [pd.read_csv(f, converters = {'codproduto': str, 'filial':str}) for f in files ]
    #df_estoque = pd.concat(intermed, ignore_index=True)
    #EMENDA
    df_estoque = pd.read_csv('df_estoque_reformat_ABC_' +s_ano + s_mes + s_dia +'.csv')
    df_estoque = df_estoque.iloc[:, 1:]
    display(df_estoque)
    # Removendo duplicatas
    df = df_estoque
    df = df.drop_duplicates()
    df = df.reset_index()
    df = df.iloc[:, 1:]
    df_estoque = df

    df_estoque_copia = df_estoque
    #! rm produto*.csv
    df = df_estoque_copia
    df = df.drop_duplicates()
    df = df.reset_index()
    df = df.iloc[:, 1:]

    #df_sduplo = df.drop_duplicates(subset='CodProduto')
    #df_source_sduplo = pd.DataFrame([flatten_json(ast.literal_eval(x)) for x in df_sduplo.estoques.to_list()])
    df_source_sduplo = df
    tamanho = len(df_source_sduplo.columns)
    tamanho
    #df_source_sduplo = df_source_sduplo.iloc[:, 1:]
    print('df_source_sduplo')
    display(df_source_sduplo)
    print(list(df_source_sduplo.IdFilial.unique()))

    #df_final = pd.DataFrame()

    #for i in range(int(tamanho/4.0)):
    #    print(i)
    #    df_aux = df_source_sduplo.iloc[:, [4*i,4*i+1,4*i+2,4*i+3]]
    #    df_aux.columns = ['IdFilial', 'armazem', 'QuantdadeItem', 'custo']
    #    df_aux['CodProduto']=list(df_sduplo.CodProduto)
    #    #display(df_aux)
    #    df_final = pd.concat([df_final, df_aux], axis=0)

    df_final_snan = df_source_sduplo.dropna()
    df = df_final_snan

    #display(df)
    #df = df[['codproduto', 'filial', 'armazem', 'quantidade']]
    #df.columns = ['CodProduto', 'IdFilial', 'armazem', 'QuantdadeItem']
    #df.armazem = df.armazem.replace(np.nan, 0.0)
    #df.armazem = df.armazem.replace('TI', 999999999.0)
    #df.armazem = df.armazem.astype('int64')

    list_filiN = [101, 104, 108, 117, '0101', '0104', '0108', '0117']
    list_a = list(df.IdFilial.unique())
    print('list_a')
    print(list_a)
    list_filiB =  list(set(list_a).difference(set(list_filiN)))
    print('list_filiB')
    print(list_filiB)
    #display(df)
    #display(df[df.IdFilial.isin(list_filiB)])
    df = df[df.IdFilial.isin(list_filiB)]
    #display(df)
    dfraw = df[df.armazem.isin([1,2,3,4,'01','02','03','04'])]
    print('dfraw')
    display(dfraw)
    #OUTPUT1
    #Salvando
    dfraw.to_csv(r'df_estoque_ABC_' + s_ano + s_mes + s_dia + '_raw.csv')
    df = dfraw.groupby(['CodProduto'])['QuantdadeItem'] .sum(True)
    df = df.reset_index()
    print('df')
    display(df)
    #OUTPUT2
    #Salvando
    df.to_csv(r'df_estoque_ABC_' + s_data + '.csv')

else:
     print(' ')
#OUTPUT3
# Salvando
df_vendas_finalMt = df_vendas_finalM.transpose()
#df_vendas_finalMt.iloc[0, 0] = df_vendas_finalMt.iloc[0, 0].date()

(df_vendas_finalMt).to_csv(r'df_hist_' + s_data + '_linha.csv')